In [1]:
import sys
from pathlib import Path
# Ensure repo imports work when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

## Notebook overview (job-readiness)

This notebook is the end-to-end demo for the repo:

- **Input**: mixed-resolution CSVs in `data/raw/`
- **Cleaning**: auto-generate standardized half-hourly CSVs in `data/cleaned/` (if missing)
- **Insights**: compute an explainable summary from signals (elec/temp/humidity/weather) + home device profile
- **LLM**: feed the summary as grounded context and answer questions
- **Plots**: visualize time series to sanity-check data and model reasoning

### 1. Quick start

1. Install deps: `pip install -r requirements.txt`
2. No pre-steps needed: if `data/cleaned/*.csv` are missing, the notebook will run `clean_to_halfhourly` automatically.
3. Run cells top-to-bottom.

### Data contract (cleaned CSVs)

- `data/cleaned/electricity.csv`: `timestamp,kwh`
- `data/cleaned/internal_temp.csv`: `timestamp,temperature_celsius`
- `data/cleaned/humidity.csv`: `timestamp,humidity_percent`
- `data/cleaned/weather.csv`: `timestamp,<weather columns...>`
- `data/cleaned/home_profile.csv`: device/fuel profile (electric vs gas flags, EV/solar/heat-pump/battery presence)

### Common gotchas

- **Import errors**: run the first cell (adds repo root to `sys.path`).
- **Empty plots/insights**: ensure `data/cleaned/` has the CSVs (the notebook auto-cleans if needed).
- **Timezones**: loaders/coercion assume UTC; keep timestamps consistent.

In [2]:
# Ensure cleaned (half-hourly) CSVs exist; if missing, run the cleaner automatically.
from pathlib import Path

from config import (
    ELECTRICITY_CSV,
    HUMIDITY_CSV,
    TEMPERATURE_CSV,
    WEATHER_CSV,
    HOME_PROFILE_CSV,
)
from data.clean_to_halfhourly import clean_all
from data.loaders import (
    load_elec,
    load_humidity,
    load_temperature,
    load_weather,
    load_home_profile,
)

_cleaned_required = [ELECTRICITY_CSV, HUMIDITY_CSV, TEMPERATURE_CSV, WEATHER_CSV]
_missing = [p for p in _cleaned_required if not Path(p).exists()]

if _missing:
    print(f"Missing cleaned CSVs: {', '.join(str(p) for p in _missing)}")
    print("Running clean_to_halfhourly to generate data/cleaned/*.csv ...")
    clean_all()

# Home profile is static; just load it.
home_profile = load_home_profile(HOME_PROFILE_CSV)

elec = load_elec(ELECTRICITY_CSV)
hum = load_humidity(HUMIDITY_CSV)
temp = load_temperature(TEMPERATURE_CSV)
weather = load_weather(WEATHER_CSV)

### 2. Build grounded context for the LLM

We compute an **explainable** context block from the cleaned data (coverage, patterns, comfort, weather, correlations) and pass it to the model. The model is instructed to use **only** this provided context when answering.

In [3]:
# Build richer insight context from all loaded data (elec/temp/hum/weather + home device profile)
from insights import (
    peak_usage_times,
    tariff_recommendation,
    schedule_suggestion,
    temperature_summary,
    humidity_summary,
    build_household_context,
)
from insights.tariff import compute_elec_cost_gbp, tariff_cost_summary

# Your tariff (pence)
TARIFF = {
    "standing_charge_p_per_day": 61.95,
    "base_rate_p_per_kwh": 20.88,      # 05:00–16:00 and 19:00–02:00
    "offpeak_rate_p_per_kwh": 15.68,   # 02:00–05:00
    "peak_rate_p_per_kwh": 38.98,      # 16:00–19:00
    # Make window times configurable directly in the notebook (local time-of-day)
    "offpeak_windows": ["02:00-05:00"],
    "peak_windows": ["16:00-19:00"],
}

parts = []

# High-level aggregated block (coverage + patterns + comfort + weather + correlations)
parts.append(
    build_household_context(
        elec=elec,
        internal_temp=temp,
        internal_humidity=hum,
        weather=weather,
        home_profile=home_profile,
        tariff_config=TARIFF,
    )
)

# Keep your original “quick hits” too (now includes cost under your tariff)
if elec.empty:
    parts.append("Peak/tariff/schedule: No electricity data available.")
else:
    peaks = peak_usage_times(elec)
    tariff = tariff_recommendation(elec)
    sched = schedule_suggestion(elec)

    # Cost under your tariff
    elec_cost = compute_elec_cost_gbp(elec, **TARIFF)
    cost_summary = tariff_cost_summary(elec_cost)

    parts.append(
        f"QUICK HITS\n"
        f"- Peak usage hours: {peaks.get('peak_hours', [])}; peak days: {peaks.get('peak_days', [])}.\n"
        f"- Tariff (generic): {tariff.get('recommendation', '')} (peak share {tariff.get('peak_share')}).\n"
        f"- Cost (your tariff): {cost_summary.get('message', '')}\n"
        f"- Schedule: {sched.get('message', '')} Best hours: {sched.get('best_hours', [])}."
    )

# Keep the simple summaries (they read nicely in answers)
temp_insight = temperature_summary(temp)
hum_insight = humidity_summary(hum)
if temp_insight.get("message") and "Not enough" not in temp_insight["message"]:
    parts.append(temp_insight["message"])
if hum_insight.get("message") and "Not enough" not in hum_insight["message"]:
    parts.append(hum_insight["message"])

insight_text = "\n\n".join([p for p in parts if p])

print(insight_text[:900], "..." if len(insight_text) > 900 else "")

DATA COVERAGE
- Electricity: 2026-01-19 10:00:00+00:00 → 2026-03-14 23:30:00+00:00 (2620/2620 points)
- Internal temperature: 2026-01-23 16:00:00+00:00 → 2026-03-15 19:00:00+00:00 (2455/2455 points)
- Internal humidity: 2026-01-23 16:00:00+00:00 → 2026-03-15 19:30:00+00:00 (2456/2456 points)
- Weather: 2025-12-15 00:00:00+00:00 → 2026-03-15 00:00:00+00:00 (4321/4321 points)

ELECTRICITY PATTERNS
- Typical daily usage: mean=6.53 kWh, p50=7.70 kWh, p90=12.59 kWh
- Peak half-hour usage: 1.78 kWh (p99), max=2.77 kWh
- Baseload estimate (p10 half-hour): 0.00 kWh
- Weekday vs weekend avg daily usage: 7.05 vs 5.13 kWh
- Highest-usage hours: [4, 15, 3] (hour-of-day)
- Lowest-usage hours: [1, 7, 0] (hour-of-day)

TARIFF SCHEDULE (YOUR RATES)
- Standing charge: 61.95p/day
- Off-peak: 02:00-05:00 @ 15.68p/kWh
- Peak: 16:00-19:00 @ 38.98p/kWh
- Base: all other times @ 20.88p/kWh
- Cheapest energy ra ...


### 3. Plot time series (internal temp, humidity, weather, electricity + cost)

Tariff used for cost line (local time-of-day):
- Standing charge: 61.95p/day
- Off-peak window (`TARIFF["offpeak_windows"]`): 02:00–05:00 @ 15.68p/kWh
- Peak window (`TARIFF["peak_windows"]`): 16:00–19:00 @ 38.98p/kWh
- Base: all other times @ 20.88p/kWh

In [4]:

# Plot half-hourly data (uses elec, hum, temp, weather from first data cell)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from functools import reduce

from insights.tariff import compute_elec_cost_gbp

indexes = [s.index for s in [elec, hum, temp] if not s.empty]
if not weather.empty:
    indexes.append(weather.index)
df = None
if indexes:
    idx = reduce(lambda a, b: a.union(b), indexes)
    idx = idx[idx.notna()].drop_duplicates().sort_values()
    df = pd.DataFrame(index=idx)
    if not elec.empty:
        df["elec_kwh"] = elec.reindex(idx)
        # Cost under your tariff (GBP)
        df["elec_cost_gbp"] = compute_elec_cost_gbp(
            elec,
            standing_charge_p_per_day=TARIFF["standing_charge_p_per_day"],
            base_rate_p_per_kwh=TARIFF["base_rate_p_per_kwh"],
            offpeak_rate_p_per_kwh=TARIFF["offpeak_rate_p_per_kwh"],
            peak_rate_p_per_kwh=TARIFF["peak_rate_p_per_kwh"],
            offpeak_windows=TARIFF["offpeak_windows"],
            peak_windows=TARIFF["peak_windows"],
        ).reindex(idx)
    if not temp.empty:
        df["internal_temperature_c"] = temp.reindex(idx)
    if not hum.empty:
        df["internal_humidity_pct"] = hum.reindex(idx)
    # Add all weather columns into df so we can plot them all.
    # Note: these columns have different units/scales, so we plot them on one row for quick comparison.
    if not weather.empty:
        weather_cols = [c for c in weather.columns if c not in ("timestamp", "ts")]
        for c in weather_cols:
            w = weather[c].copy()
            if not w.index.is_unique:
                w = w.loc[~w.index.duplicated(keep="first")]
            df[c] = w.reindex(idx)
    df = df.dropna(how="all").sort_index()

if df is None or df.empty:
    print("No data to plot. Ensure cleaned CSVs exist (notebook will auto-clean at the top).")
else:
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        subplot_titles=(
            "Internal temperature (°C)",
            "Internal humidity (%)",
            "Weather variables",
            "Electricity (kWh) + cost (GBP)",
        ),
        vertical_spacing=0.06,
        specs=[[{}], [{}], [{}], [{"secondary_y": True}]],
    )

    x = df.index

    # Internal temperature
    if "internal_temperature_c" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["internal_temperature_c"], name="Internal temp", line=dict(color="#1f77b4")),
            row=1,
            col=1,
        )

    # Internal humidity
    if "internal_humidity_pct" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["internal_humidity_pct"], name="Internal humidity", line=dict(color="#2ca02c")),
            row=2,
            col=1,
        )

    # Row 3: all weather variables
    weather_cols = [c for c in weather.columns if c not in ("timestamp", "ts")]
    for c in weather_cols:
        if c in df.columns:
            fig.add_trace(
                go.Scatter(x=x, y=df[c], name=c, mode="lines"),
                row=3,
                col=1,
            )

    # Electricity usage + cost
    if "elec_kwh" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["elec_kwh"], name="Elec (kWh)", line=dict(color="#d62728")),
            row=4,
            col=1,
            secondary_y=False,
        )
    if "elec_cost_gbp" in df.columns:
        fig.add_trace(
            go.Scatter(x=x, y=df["elec_cost_gbp"], name="Cost (£)", line=dict(color="#9467bd")),
            row=4,
            col=1,
            secondary_y=True,
        )

    fig.update_layout(height=650, title_text="Household energy and environment", showlegend=True)
    fig.update_yaxes(title_text="°C", row=1, col=1)
    fig.update_yaxes(title_text="%", row=2, col=1)
    fig.update_yaxes(title_text="Weather values", row=3, col=1)
    fig.update_yaxes(title_text="kWh", row=4, col=1, secondary_y=False)
    fig.update_yaxes(title_text="£", row=4, col=1, secondary_y=True)
    fig.show()

In [5]:
# Load HF model (uses M2 GPU via MPS, or CUDA/CPU)
import torch
from transformers import pipeline

if torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon (M1/M2/M3) GPU
elif torch.cuda.is_available():
    device = 0
else:
    device = -1  # CPU
print(f"Device: {device}" + (" (Apple M-series GPU)" if device == "mps" else (" (NVIDIA GPU)" if device != -1 else " (CPU)")))

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    dtype=torch.float32 if device == -1 else torch.float16,
    device=device,
)
print(f"Model on: {next(pipe.model.parameters()).device}")

Device: mps (Apple M-series GPU)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model on: mps:0


In [6]:
# 4b. Run golden evals (groundedness + required phrases)
import json
import re
from pathlib import Path

from evals.run_evals import evaluate_case

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVALS_DIR = ROOT / "evals"
GOLDEN_PATH = EVALS_DIR / "golden_qa.jsonl"

if not GOLDEN_PATH.exists():
    raise FileNotFoundError(f"Missing golden eval file: {GOLDEN_PATH}")

# Same strict context contract used in the interactive Q&A
eval_context = (
    "You are an energy advisor. The text below is THIS household's energy data.\n"
    "Use ONLY this data to answer.\n"
    "If the answer is NOT explicitly present in the context, reply EXACTLY with: "
    "'Not enough data in context to answer.'\n"
    "Do not invent numbers or facts.\n\n"
    "Output format (follow exactly):\n"
    "Answer: <concise answer>\n"
    "Evidence: <copy 1-3 short relevant context lines>\n\n"
    "--- HOUSEHOLD DATA ---\n"
    f"{insight_text}\n"
    "--- END DATA ---"
)

# Deterministic generation for eval stability
gen_kwargs_eval = {
    "max_new_tokens": 250,
    "do_sample": False,
    "pad_token_id": pipe.tokenizer.eos_token_id,
}

cases = []
for line in GOLDEN_PATH.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    cases.append(json.loads(line))

max_retries = 3
results = []

for case in cases:
    feedback = None
    last_res = None
    for _attempt in range(max_retries):
        content = f"{eval_context}\n\nQuestion: {case['question']}"
        if feedback:
            content += f"\n\n{feedback}\n\nRegenerate now. Ensure you explicitly include the required phrases when they are present in the context."

        messages = [{"role": "user", "content": content}]
        prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        out = pipe(prompt, **gen_kwargs_eval)[0]["generated_text"]

        # Strip prompt echo
        if out.startswith(prompt):
            out = out[len(prompt):].strip()

        last_res = evaluate_case(case=case, answer=out.strip(), context=eval_context)
        if last_res.passed:
            break

        must_any = case.get("must_include_any", [])
        must_not = case.get("must_not_include_any", [])
        reasons = "\n".join([f"- {r}" for r in last_res.reasons]) if last_res.reasons else "(no reasons)"
        feedback = (
            "Constraint feedback (your previous answer failed checks):\n"
            f"{reasons}\n"
            f"Required phrases (include at least one): {must_any}\n"
            f"Forbidden phrases (include none): {must_not}"
        )

    assert last_res is not None
    results.append(last_res)

passed = sum(1 for r in results if r.passed)
print(f"Golden evals: {passed}/{len(results)} passed")
for r in results:
    status = "PASS" if r.passed else "FAIL"
    print(f"- {status} {r.case_id}")
    if not r.passed:
        for reason in r.reasons:
            print(f"  - {reason}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set

Golden evals: 4/4 passed
- PASS tariff_reco
- PASS comfort_summary
- PASS weather_effect
- PASS schedule_shift


### 4. Interactive Q&A
Run the cell below once. A text box and **Ask** button will appear in the notebook. Type your question and click **Ask**; the reply appears below (no terminal needed).

The assistant reply follows a strict format:
- `Answer: ...`
- `Evidence: ...` (short context lines)

If the answer isn’t explicitly in the provided context, it replies exactly:
`Not enough data in context to answer.`

This strict contract is also used by `evals/run_evals.py`.

In [7]:
# Interactive Q&A: ask follow-ups without re-running the model (type 'q' to quit)
import textwrap
from transformers import GenerationConfig

gen_config = GenerationConfig(
    max_new_tokens=150,
    do_sample=True,
    temperature=0.3,
    pad_token_id=pipe.tokenizer.eos_token_id,
)
# Make it explicit that the data is provided in the message (so the model doesn't say it lacks access)
context = (
    "You are an energy advisor. The text below is THIS household's energy data. "
    "Use ONLY this data to answer.\n\n"
    "--- HOUSEHOLD DATA ---\n"
    f"{insight_text}\n"
    "--- END DATA ---"
)

# Use a text box + button so input appears in the notebook (no terminal)
from ipywidgets import Text, Button, Output, VBox

txt = Text(placeholder="e.g. When should I run my washing machine?", description="Question:", style={"description_width": "80px"}, layout={"width": "500px"})
out_area = Output()
btn = Button(description="Ask")

def on_ask(_):
    q = txt.value.strip()
    if not q:
        return
    with out_area:
        print(f"You: {q}")
        messages = [{"role": "user", "content": f"{context}\n\nQuestion: {q}\n\nOutput format (follow exactly):\nAnswer: <concise answer>\nEvidence: <copy 1-3 short relevant context lines>\nIf the answer is NOT explicitly present in the context, reply EXACTLY: Not enough data in context to answer."}]
        prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        reply = pipe(prompt, generation_config=gen_config)[0]["generated_text"]
        if prompt in reply:
            reply = reply[len(prompt):].strip()
        # Keep full structured output; only strip leading chat marker lines.
        reply = reply.strip()
        lines = reply.splitlines()
        if lines:
            first = lines[0].strip().lower()
            if first.startswith("assistant"):
                # Drop the first line if it is only a marker (common with chat templates).
                if len(lines) > 1:
                    reply = "\n".join(lines[1:]).strip()

        print(textwrap.fill(reply or "(no reply)", width=72), "\n")

btn.on_click(on_ask)
display(VBox([txt, btn, out_area]))